In [32]:
from yahoo_fin import stock_info as si

def get_tickers():
    return si.tickers_sp500()

In [33]:
def add_trend_signal(df):
    df['Trend'] = 0
    for i in range(5, len(df)):
        window = df.iloc[i-5:i]
        if all(window['Close'] > window['EMA_200']):
            df.loc[i, 'Trend'] = 1
        elif all(window['Close'] < window['EMA_200']):
            df.loc[i, 'Trend'] = -1

    return df

In [34]:
def add_macd_signal(df):
    df['MACD_cross'] = 0
    for i in range(2, len(df)):
        before = df.iloc[i-2:i]
        if all(before['MACD_12_26_9'] < before['MACDs_12_26_9']) and df.loc[i, 'MACD_12_26_9'] > df.loc[i, 'MACDs_12_26_9']:
            df.loc[i, 'MACD_cross'] = 1
        elif all(before['MACD_12_26_9'] > before['MACDs_12_26_9']) and df.loc[i, 'MACD_12_26_9'] < df.loc[i, 'MACDs_12_26_9']:
            df.loc[i, 'MACD_cross'] = -1

    return df
            

In [35]:
def add_sma_signal(df):
    df['SMA_cross'] = 0
    for i in range(5, len(df)):
        window = df.iloc[i-5:i]
        if all(window['SMA_50'] < window['SMA_200']):
            if df.loc[i, 'SMA_50'] > df.loc[i, 'SMA_200']:
                df.loc[i, 'SMA_cross'] = 1
        elif all(window['SMA_50'] > window['SMA_200']):
            if df.loc[i, 'SMA_50'] < df.loc[i, 'SMA_200']:
                df.loc[i, 'SMA_cross'] = -1

    return df

In [36]:
def add_rsi_signal(df):
    df['rsi_signal'] = 0
    for i in range(1, len(df)):
        if (df.loc[i - 1, 'RSI'] > 70) & (df.loc[i, 'RSI'] < 70):
            df.loc[i, 'rsi_signal'] = -1
        elif (df.loc[i - 1, 'RSI'] < 30) & (df.loc[i, 'RSI'] > 30):
            df.loc[i, 'rsi_signal'] = 1

    return df

In [37]:
# def add_total_signal(df):
    # Strategy: Trade with trend, rsi > 50, and engulfing candle
    # Trade Management: Stop loss @ 2x candle width, 1.5 risk to reward ratio
    # -------------------------------------------------------------------------------
    # Trial 1: Very negative returns, going to try just buys
    # Trial 2: Only did buy positions, positive returns, not as high as buy & hold returns
    # Results: ~35% win rate with positive decent returns (14.5%) for 2.0 ratio
    
    # df['engulfing_signal'] = 0
    # df.loc[ (df['Close'] > df['EMA_200']) & (df['RSI'] > 50) & (df['Engulfing'] == 100), 'engulfing_signal'] = 1

    # Strategy: Trade with trend, if candle closes outside of bollinger bands, make a limit order at 3% below closing price
    # Trade Management: Limit order is valid for 5 days, stop loss @ 3% below limit, 1.5 risk to reward ratio
    # -------------------------------------------------------------------------------
    # Trial 1: ~40% win rate with positive decent returns (11.5%) for 1.9 ratio
    # Trail 2: Sell when RSI crosses 50 line, much better returns (~27%)
    # Trial 3: Same trade management, only buy signals. ~45% return rate and 72% win rate
    # df['bb_signal'] = 0
    # df.loc[ (df['Trend'] == 1) & (df['Close'] < df['BBL_20_2.0']), 'bb_signal'] = 1
    # df.loc[ (df['Trend'] == -1) & (df['Close'] > df['BBU_20_2.0']), 'bb_signal'] = -1

    # Strategy: Trade with trend, look for MACD crossover above or below the zero line
    # Trade Management: Optimize parameters for sl and tp
    # -------------------------------------------------------------------------------
    # Trial 1: ~ 60% win rate with ~16% return rate, begin paper trade testing
    # df['macd_signal'] = 0
    # df.loc[ (df['MACD_cross'] == 1) & (df['RSI'] < 30), 'macd_signal'] = 1
    # df.loc[ (df['MACD_cross'] == -1) & (df['RSI'] > 70), 'macd_signal'] = -1

    #Strategy: If RSI crosses down from above 70 sell, up from above 30 buy
    # Trade Management: 1.5% stop loss, 3% take profit
    # --------------------------------------------------------------------------------


    # return df

In [38]:
# import yfinance as yf
# from yahoo_fin import stock_info as si
# import pandas_ta as ta
# import numpy as np
# import pandas as pd

# def import_data(stock, startDate, intvl):
#     data = yf.download(stock, start=startDate, interval=intvl)

#     data['EMA_200'] = ta.ema(data['Close'], length=200)
#     # data['SMA_50'] = ta.sma(data['Close'], length=50)
#     # data['SMA_200'] = ta.sma(data['Close'], length=200)
#     data['RSI'] = ta.rsi(data['Close'], length=14)
#     data['ATR'] = ta.atr(data['High'], data['Low'], data['Close'], length=14)
#     # data['Engulfing'] = data.ta.cdl_pattern(name="engulfing")
#     bbands = ta.bbands(data['Close'], length=20, std=2)
#     macd = ta.macd(data['Close'])
#     data = data.join(bbands)
#     data = data.join(macd)
#     ichimoku = ta.ichimoku(data['High'], data['Low'], data['Close'])
#     data = data.join(ichimoku[0])
#     # data = data.join(ichimoku[1])
    
#     data.reset_index(inplace=True)
    
#     data = add_trend_signal(data)
#     data = add_macd_signal(data)
#     data = add_total_signal(data)
#     # data = add_sma_signal(data)
#     data = add_rsi_signal(data)

#     return data

In [39]:
import yfinance as yf
import pandas_ta as ta
import pandas as pd
import numpy as np

def import_data(stock, startDate, intvl):
    # Download data from yfinance
    data = yf.download(stock, start=startDate, interval=intvl)
    
    # Add technical indicators
    data['MA_50'] = ta.sma(data['Close'], length=50)  # 50-period MA
    data['MA_200'] = ta.sma(data['Close'], length=200)  # 200-period MA
    data['RSI'] = ta.rsi(data['Close'], length=14)  # 14-period RSI
    data['RSI_2'] = ta.rsi(data['Close'], length=2)  # 14-period RSI
    data['ATR'] = ta.atr(data['High'], data['Low'], data['Close'], length=14)  # 14-period ATR for stop loss and trailing
    
    data.dropna(inplace=True)  # Drop any rows with NaN values due to indicators

    data = data.reset_index(drop=True)

    return data


In [40]:
def add_total_signal(df):
    df['rsi_signal'] = 0  # Signal column to store buy/sell signals

    for i in range(3, len(df)):
        # Buy signal: 50-day MA crosses above 200-day MA (Golden Cross) and RSI is between 40 and 60
        if (df.loc[i, 'Close'] > df.loc[i, 'MA_200']) and (df.loc[i, 'RSI_2'] <  5):
            df.loc[i, 'rsi_signal'] = 1  # Buy signal

    return df


In [41]:
from backtesting import Strategy, Backtest

def get_parameters(df):
    # Define the signal function that will be used in the strategy
    def SIGNAL():
        return df['rsi_signal']
    
    class MA_Crossover_RSI_Strategy(Strategy):
        mysize = 0.1  # Trade size (10% of available cash)
        
        # Define global variables for stop loss and take profit
        # atr_multiplier = 1.5  # Multiplier for ATR-based stop loss
        # trailing_multiplier = 2  # Multiplier for ATR-based trailing stop

        def init(self):
            # Initialize the signal and ATR for stop loss and trailing stop calculation
            self.signal1 = self.I(SIGNAL)
            self.rsi2 = self.I(lambda: df['RSI_2'], name="RSI")

        def next(self):
            # If no open position and a buy signal occurs, enter a buy order
            if self.signal1 == 1 and not self.position:
                self.buy(size=self.mysize)
                # print(f"BUY at {entry_price}, Stop Loss at {stop_loss}")

            # Optional: Use RSI reaching 80 as a take profit signal
            if self.position:
                if self.position.is_long and self.rsi2[-1] > 70:
                    self.position.close()
                # print(f"RSI Exit at {self.data.Close[-1]} (RSI >= 80)")

    # Run the backtest
    bt = Backtest(df, MA_Crossover_RSI_Strategy, cash=10000, margin=1/30, commission=0.01)
    results = bt.run()

    return results

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def print_candles(df):
    # Create a figure with two rows for subplots
    fig = make_subplots(rows=1, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.3, 
                        row_heights=[0.7],
                        subplot_titles=("Candlestick Chart"))

    # Candlestick chart
    fig.add_trace(go.Candlestick(x=df.index,
                                 open=df['Open'],
                                 high=df['High'],
                                 low=df['Low'],
                                 close=df['Close']),
                  row=1, col=1)

    # EMA and Bollinger Bands on the first plot
    # fig.add_trace(go.Scatter(x=df.index, y=df['SMA_50'], 
    #                          line=dict(color='red', width=1), 
    #                          name="EMA 200"),
    #               row=1, col=1)
    # fig.add_trace(go.Scatter(x=df.index, y=df['SMA_200'], 
    #                          line=dict(color='blue', width=1), 
    #                          name="BB Upper"),
    #               row=1, col=1)

    # Buy and sell signals on the first plot
    buys = df.loc[(df['rsi_signal'] == 1)].index
    sells = df.loc[(df['rsi_signal'] == -1)].index

    for date in buys:
        fig.add_vline(x=date, line_width=1, line_dash="dash", line_color="green", row=1, col=1)
    for date in sells:
        fig.add_vline(x=date, line_width=1, line_dash="dash", line_color="red", row=1, col=1)

    # RSI on the second plot
    fig.add_trace(go.Scatter(x=df.index, y=df['RSI_2'], 
                             line=dict(color='blue', width=1), 
                             name="RSI"),
                  row=2, col=1)
    # fig.add_trace(go.Scatter(x=df.index, y=df['MACDs_12_26_9'], 
    #                          line=dict(color='orange', width=1), 
    #                          name="RSI"),
    #               row=2, col=1)

    # Update layout
    fig.update_layout(width=1200, height=800)
    fig.show()

In [43]:
# from backtesting import Strategy
# from backtesting import Backtest
# def get_parameters(df):
#     def SIGNAL():
#         return df['bb_signal']
    
#     class MyStrat(Strategy):
#         mysize = 0.1
#         order_duration = 0
#         max_order_duration = 7
#         limit_order = None

#         def init(self):
#             super().init()
#             self.signal1 = self.I(SIGNAL)

#         def next(self):
#             super().next()
#             if self.limit_order:
#                 self.order_duration += 1

#                 if self.order_duration > self.max_order_duration:
#                     self.limit_order.cancel

#             if self.signal1==1 and len(self.trades)==0:
#                 if len(self.orders) == 0: 
#                     order = self.data.Close[-1] - self.data.Close[-1] * 0.03
#                     stop_loss = order - self.data.Close[-1] * 0.03
#                     num_shares = int((self.mysize * self.equity) / (order - stop_loss))
#                     self.limit_order = self.buy(limit=order, size=num_shares)
#                     self.order_duration = 0
            
#             # elif self.signal1==-1 and len(self.trades)==0:         
#             #     if len(self.orders) == 0: 
#             #         stop_loss = self.data.Close[-1] + self.data.Close[-1] * 0.02
#             #         take_profit = self.data.Close[-1] - self.data.Close[-1] * 0.03
#             #         num_shares = int((self.mysize * self.equity) / (stop_loss - self.data.Close[-1]))
#             #         self.sell(sl=stop_loss, tp=take_profit, size=num_shares)
#             if len(self.trades) > 0:
#                 if self.data.RSI[-1] > 50:
#                     self.position.close

#     bt = Backtest(df, MyStrat, cash=10000, margin=1/30)
#     results = bt.run()
#     # print(results)
#     # print(results._trades)

#     return results

In [44]:
# Strategy #3 Backtest
from backtesting import Strategy
from backtesting import Backtest
def optimize(df):
    def SIGNAL():
        return df['macd_signal']
    
    class MyStrat(Strategy):
        mysize = 0.1
        sl = 1.5
        ratio = 2

        def init(self):
            super().init()
            self.signal1 = self.I(SIGNAL)

        def next(self):
            super().next()        
            if self.signal1==1 and len(self.trades)==0:
                sl_value = self.sl * self.data.ATR[-1]
                stop_loss = self.data.Close[-1] - sl_value
                take_profit = self.data.Close[-1] + (sl_value * self.ratio)
                self.buy(sl=stop_loss, tp=take_profit, size=self.mysize)
            
            elif self.signal1==-1 and len(self.trades)==0:         
                sl_value = self.sl * self.data.ATR[-1]
                stop_loss = self.data.Close[-1] + sl_value
                take_profit = self.data.Close[-1] - (sl_value * self.ratio)
                self.sell(sl=stop_loss, tp=take_profit, size=self.mysize)

    bt = Backtest(df, MyStrat, cash=10000, margin=1/30)
    # results = bt.run()
    # print(results)
    # print(results._trades)

    stats, heatmap = bt.optimize(sl=[i/10 for i in range(10, 26)],
                        ratio=[i/10 for i in range(10, 21)],
                        maximize='Return [%]', max_tries=300,
                            random_state=0,
                            return_heatmap=True)
    # print(stats)
    # print(stats['_strategy'])
    # print(stats['_trades'])

    return stats['_strategy']

In [45]:
import math

def test_full_strategy():
    # for ratio in range(10, 30):
        my_results = []
        buy_hold = []
        wins = []
        trades = []
        sp500_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
        sp500_table = pd.read_html(sp500_url)
        TICKERS = sp500_table[0]['Symbol'].tolist()
        print(TICKERS)
        print(len(TICKERS))
        for ticker in TICKERS:
            try:
                stock = ticker
                startDate = '2019-01-01'
                interval = '1d'  # Using 1-hour timeframe

                # Import the data
                df = import_data(stock, startDate, interval)
                
                # Add trend signals (buy/sell signals based on divergence)
                df = add_total_signal(df)
                
                # Run the backtest
                results = get_parameters(df)

                my_results.append(results['Return [%]'])
                buy_hold.append(results['Buy & Hold Return [%]'])
                wins.append(results['Win Rate [%]'] * len(results['_trades']))
                trades.append(len(results['_trades']))
            except Exception as e:
                print(ticker, e)

        print(f'My average results: {sum(my_results) / len(my_results)} \n')
        print(f'Buy and Hold average results: {sum(buy_hold) / len(buy_hold)} \n')
        # win_rates = [rate for rate in win_rates if rate is not None and not math.isnan(rate)]
        print(f'Win Rate: {sum(wins) / sum(trades)} \n \n')

        # with open('equity.txt', 'a') as f:
        #     f.write(f'My average results: {sum(my_results) / len(my_results)} \n')
        #     f.write(f'Buy and Hold average results: {sum(buy_hold) / len(buy_hold)} \n')
        #     win_rates = [rate for rate in win_rates if rate is not None and not math.isnan(rate)]
        #     f.write(f'Win Rate: {sum(win_rates) / len(win_rates)} \n \n')

In [46]:
stock = 'SPY'
startDate = '2000-01-01'
interval = '1d'  # Using 1-hour timeframe

# Import the data
df = import_data(stock, startDate, interval)

df = add_total_signal(df)

# Run the backtest
results = get_parameters(df)
print(results)
print(results['_trades'])
print_candles(df)
# print(df[df['rsi_signal'] != 0])
# test_full_strategy()

[*********************100%%**********************]  1 of 1 completed
C:\Users\18053\AppData\Local\Temp\ipykernel_28324\994686794.py:33: UserWarning:

Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.



Start                                     0.0
End                                    6042.0
Duration                               6042.0
Exposure Time [%]                    7.661757
Equity Final [$]                     3230.345
Equity Peak [$]                  10687.809384
Return [%]                          -67.69655
Buy & Hold Return [%]              318.265031
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Max. Drawdown [%]                  -72.476906
Avg. Drawdown [%]                  -20.256012
Max. Drawdown Duration                 3639.0
Avg. Drawdown Duration                  877.0
# Trades                                 85.0
Win Rate [%]                        49.411765
Best Trade [%]                       3.234875
Worst Trade [%]                     -9.427929
Avg. Trade [%]                    